# AutoFeatureRec: автоматический поиск исторических признаков для рекомендаций KION

**Цель исследования**

Исследовать эффективность автоматической генерации исторических признаков для рекомендательных систем на датасете KION.

Основной исследовательский вопрос: могут ли автоматически сгенерированные признаки обеспечить качество ранжирования, сопоставимое с ручным feature engineering, и насколько эффективны различные стратегии их отбора?

**Методология**

В исследовании используется CatBoost и несколько подходов к построению признакового пространства:

* Ручное проектирование пользовательских и контентных признаков.

* Автоматическая генерация временных агрегатов на основе статистик взаимодействий.

* Отбор наиболее информативных признаков по результатам валидации.

* Жадный поиск семейств признаков (Feature-family search).

* Использование поведенческих характеристик просмотров.

Эксперименты проводятся на датасете KION с временным разделением на train, validation и test. Для воспроизводимости используется фиксированный seed=4049.

Качество оценивается с помощью ROC-AUC, LogLoss и NDCG@10. Дополнительно проводятся bootstrap-анализ, оценка вычислительных затрат и тестирование на более сложных отрицательных кандидатах, сформированных с учётом популярности контента.

**Экспериментальные принципы**

В рамках каждого протокола все сравниваемые модели оцениваются на одинаковых кандидатах. Отбор признаков и настройка моделей осуществляются по validation, а итоговое качество оценивается на test.

При построении исторических признаков используются взаимодействия, доступные до соответствующего временного среза.

**Ограничения**

Данные KION содержат агрегированные записи взаимодействий: last_watch_dt отражает дату последнего просмотра, а не полную последовательность событий.

Отрицательные примеры формируются искусственно и не соответствуют реальным показам без взаимодействия. Поэтому полученные метрики характеризуют качество офлайн-ранжирования на сформированных наборах кандидатов, а не реальный CTR.

Эксперименты с разными способами формирования отрицательных кандидатов рассматриваются как отдельные протоколы оценки. Кроме того, последовательное исследование нескольких конфигураций ограничивает независимость итоговой тестовой проверки.

## 1. Зависимости и параметры

In [ ]:
!pip -q install catboost pyarrow

In [ ]:
from pathlib import Path
from datetime import timedelta
from collections import defaultdict
import os, time, random, zipfile, urllib.request, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, log_loss
from IPython.display import display

SEED = 4049
N_USERS = 15000
NEG_PER_POS = 15
MAX_POS_PER_USER = 4         # ограничение количества положительных кандидатов на пользователя/срез
WINDOWS = [3, 7, 14, 30, 60, 120]
BASE_WINDOWS = [7, 30]
TRAIN_ITERATIONS = 200
OUT = Path('/content/autofeaturerec_final_2026'); OUT.mkdir(exist_ok=True)
DATA = Path('/content/kion_data'); DATA.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('Параметры:', {'users':N_USERS, 'neg_per_pos':NEG_PER_POS, 'windows':WINDOWS})

Параметры: {'users': 15000, 'neg_per_pos': 15, 'windows': [3, 7, 14, 30, 60, 120]}


## 2. Получение KION и обзор схемы
Оригинальные файлы: [официальный репозиторий](https://github.com/irsafilo/KION_DATASET).

In [ ]:
archive = DATA / 'data_original.zip'
urls = [
    'https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip',
    'https://raw.githubusercontent.com/irsafilo/KION_DATASET/main/data_original.zip',
    'https://github.com/irsafilo/KION_DATASET/raw/refs/heads/main/data_original.zip',
]
if not archive.exists() or not zipfile.is_zipfile(archive):
    if archive.exists(): archive.unlink()
    for url in urls:
        try:
            print('Загружаем:', url)
            urllib.request.urlretrieve(url, archive)
            if not zipfile.is_zipfile(archive): raise ValueError('Получен не ZIP')
            break
        except Exception as e:
            print('Попытка не удалась:', str(e)[:170])
            if archive.exists(): archive.unlink()
    else:
        raise RuntimeError('Не получилось загрузить ZIP. Откройте GitHub и загрузите data_original.zip вручную в /content/kion_data/')
with zipfile.ZipFile(archive) as z:
    print('В архиве:', z.namelist()[:12]); z.extractall(DATA)
files = {p.name.lower():p for p in DATA.rglob('*.csv')}
assert all(k in files for k in ('interactions.csv','users.csv','items.csv')), list(files)
interactions = pd.read_csv(files['interactions.csv'], parse_dates=['last_watch_dt'])
users = pd.read_csv(files['users.csv'])
items = pd.read_csv(files['items.csv'])
print('Размеры:', interactions.shape, users.shape, items.shape)
print('Колонки interactions:', interactions.columns.tolist())
display(interactions.head(3)); display(items.head(2))

Загружаем: https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip
В архиве: ['data_original/', 'data_original/interactions.csv', '__MACOSX/data_original/._interactions.csv', 'data_original/users.csv', '__MACOSX/data_original/._users.csv', 'data_original/items.csv', '__MACOSX/data_original/._items.csv']
Размеры: (5476251, 5) (840197, 5) (15963, 14)
Колонки interactions: ['user_id', 'item_id', 'last_watch_dt', 'total_dur', 'watched_pct']


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0


,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."


## 3. Сэмплирование пользователей: полная история каждого выбранного пользователя

In [ ]:
selected = np.random.default_rng(SEED).choice(interactions['user_id'].unique(), size=min(N_USERS, interactions['user_id'].nunique()), replace=False)
events = interactions.loc[interactions['user_id'].isin(selected), ['user_id','item_id','last_watch_dt','total_dur','watched_pct']].copy()
events = events.rename(columns={'last_watch_dt':'ts'}).dropna(subset=['ts'])
events['ts'] = pd.to_datetime(events['ts'])
events = events.sort_values(['ts','user_id','item_id']).reset_index(drop=True)
events = events.drop_duplicates(['user_id','item_id','ts'])
print('События:',len(events),'пользователи:',events.user_id.nunique(),'объекты:',events.item_id.nunique())
print('Период:', events.ts.min(), '—', events.ts.max())
print('Средняя история:',round(len(events)/events.user_id.nunique(),2))
assert events.ts.notna().all()
del interactions

События: 86268 пользователи: 15000 объекты: 5823
Период: 2021-03-13 00:00:00 — 2021-08-22 00:00:00
Средняя история: 5.75


## 4. Категории фильмов и временные срезы
Жанр — статическая метаинформация, доступная до целевого взаимодействия. Train = первые 2 среза, validation = третий, test = четвёртый.

In [ ]:
assert 'item_id' in items.columns
meta = items[['item_id']].copy()
if 'genres' in items.columns:
    meta['genre'] = items['genres'].fillna('unknown').astype(str).str.split(',').str[0].str.strip().replace('', 'unknown')
else:
    meta['genre'] = 'unknown'
meta = meta.drop_duplicates('item_id')
# границы по календарному времени, а не по строкам
start, finish = events.ts.min(), events.ts.max()
fracs = [0.40,0.55,0.70,0.85,1.0]
bounds = [start + (finish-start)*f for f in fracs]
print('Срезы:', [(str(a.date()),str(b.date())) for a,b in zip(bounds[:-1],bounds[1:])])
assert len(set(bounds)) == 5, 'Очень короткий временной диапазон'
# хранение только нужных полей уменьшает память
events = events.merge(meta, on='item_id', how='left')
events['genre'] = events['genre'].fillna('unknown')
assert events['ts'].is_monotonic_increasing


Срезы: [('2021-05-16', '2021-06-10'), ('2021-06-10', '2021-07-04'), ('2021-07-04', '2021-07-28'), ('2021-07-28', '2021-08-22')]


## 5. Генерация кандидатов без заглядывания в будущее
На срезе каждый пользователь с хотя бы одной новой активностью получает положительные объекты в следующем окне и до `NEG_PER_POS` случайных кандидатов на каждый положительный. Каталог и частоты для выборки кандидатов определяются **только по прошлому**; не исключаем будущие просмотры за пределами оцениваемого окна.

In [ ]:
def make_candidates(history, future, snap_id, max_negs=NEG_PER_POS):
    rng = np.random.default_rng(SEED + snap_id)
    catalog = history['item_id'].drop_duplicates().to_numpy()
    if len(catalog) < 30:
        raise ValueError('Каталог в истории слишком маленький')
    # ограничиваем целевых пользователей теми, для кого есть история на момент среза
    known_users = set(history['user_id'].unique())
    active = future[future.user_id.isin(known_users)]
    seen = history.groupby('user_id')['item_id'].agg(set).to_dict()
    available = set(catalog)
    positives = (active[active.item_id.isin(available)]
                 .drop_duplicates(['user_id','item_id'])
                 .sort_values(['user_id','ts']))
    rows = []
    for uid, group in positives.groupby('user_id', sort=False):
        pos_ids = group['item_id'].to_numpy()[:MAX_POS_PER_USER]
        pos_set = set(group.item_id.tolist())  # предотвращаем false negatives для окна
        blocked = seen[uid] | pos_set
        rows.extend((snap_id, uid, int(i), 1) for i in pos_ids)
        eligible = np.array([i for i in catalog if i not in blocked], dtype=catalog.dtype)
        nneg = min(len(eligible), max_negs * len(pos_ids))
        if nneg:
            negs = rng.choice(eligible, size=nneg, replace=False)
            rows.extend((snap_id, uid, int(i), 0) for i in negs)
    result = pd.DataFrame(rows, columns=['snapshot','user_id','item_id','label'])
    result = result.merge(meta, on='item_id', how='left')
    result['genre'] = result['genre'].fillna('unknown')
    return result

snapshots = []
for snap_id, (cutoff, next_cutoff) in enumerate(zip(bounds[:-1],bounds[1:])):
    hist = events[events.ts < cutoff]
    fut = events[(events.ts >= cutoff) & (events.ts < next_cutoff if snap_id < 3 else events.ts <= next_cutoff)]
    cand = make_candidates(hist, fut, snap_id)
    assert not cand.duplicated(['snapshot','user_id','item_id']).any()
    assert cand.label.nunique() == 2, f'Нет обоих классов в срезе {snap_id}'
    snapshots.append(cand)
    print(f'{snap_id}: cutoff={cutoff.date()}, history={len(hist)}, future={len(fut)}, candidates={len(cand)}, positives={cand.label.sum()}, users={cand.user_id.nunique()}')


0: cutoff=2021-05-16, history=19399, future=11056, candidates=46272, positives=2892, users=1216
1: cutoff=2021-06-10, history=30455, future=16561, candidates=80512, positives=5032, users=2030
2: cutoff=2021-07-04, history=47016, future=16729, candidates=83936, positives=5246, users=2258
3: cutoff=2021-07-28, history=63745, future=22523, candidates=105232, positives=6577, users=2740


## Признаки: ручной baseline и автоматическое пространство поиска

Ручные признаки — исторические `count/nunique` по пользователю, фильму, жанру и паре пользователь–жанр (вся история, 7 и 30 дней). Автоматические признаки дополнительно включают окна 3/14/60/120 дней, статистики `sum/mean/std` по длительности и проценту просмотра, давность последнего события и долю жанра в истории пользователя. Все статистики строятся **строго по `ts < cutoff`**.

Важное ограничение: поскольку KION хранит агрегированную запись с датой последнего просмотра, поведенческие значения из такой записи нельзя гарантированно восстановить на более раннем срезе. Мы используем их лишь для записей с `last_watch_dt < cutoff` и явно фиксируем это допущение.

In [ ]:
KEYS = {'user':['user_id'], 'item':['item_id'], 'genre':['genre'], 'user_genre':['user_id','genre']}

def add_aggs(frame, hist, prefix, extended=False):
    out=frame
    for entity,keys in KEYS.items():
        gb=hist.groupby(keys,observed=True,dropna=False)
        counts=gb['item_id'].agg(['size','nunique']).reset_index().rename(columns={
            'size':f'{prefix}_{entity}_cnt','nunique':f'{prefix}_{entity}_nunique'})
        out=out.merge(counts,on=keys,how='left',validate='many_to_one')
        if extended:
            stats=gb.agg(dur_sum=('total_dur','sum'),dur_mean=('total_dur','mean'),
                         dur_std=('total_dur','std'),pct_mean=('watched_pct','mean'),
                         pct_std=('watched_pct','std'),last_ts=('ts','max')).reset_index()
            stats[f'{prefix}_{entity}_recency_days']=(CURRENT_CUTOFF-stats['last_ts']).dt.total_seconds()/86400
            stats=stats.drop(columns='last_ts').rename(columns={k:f'{prefix}_{entity}_{k}' for k in ['dur_sum','dur_mean','dur_std','pct_mean','pct_std']})
            out=out.merge(stats,on=keys,how='left',validate='many_to_one')
    if extended:
        num=f'{prefix}_user_genre_cnt'; den=f'{prefix}_user_cnt'
        out[f'{prefix}_user_genre_share']=out[num].fillna(0)/(out[den].fillna(0)+1e-6)
    return out

def create_features(candidates,cutoff,full_history):
    global CURRENT_CUTOFF
    CURRENT_CUTOFF=cutoff
    t0=time.monotonic()
    hist=full_history.loc[full_history.ts < cutoff]
    x=add_aggs(candidates.copy(),hist,'all',extended=True)
    for days in WINDOWS:
        h=hist.loc[hist.ts >= cutoff-pd.Timedelta(days=days)]
        x=add_aggs(x,h,f'w{days}',extended=True)
    numeric=[c for c in x.columns if c not in ['snapshot','user_id','item_id','label','genre']]
    x[numeric]=x[numeric].replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')
    x['genre']=x['genre'].astype(str)
    assert len(x)==len(candidates)
    assert (hist.ts<cutoff).all()
    return x,time.monotonic()-t0

assert {'total_dur','watched_pct'}.issubset(events.columns), 'Поведенческие поля потеряны при подготовке данных'
assert all(not c.duplicated(['snapshot','user_id','item_id']).any() for c in snapshots)
features=[]; costs=[]
for sid,(cand,cutoff) in enumerate(zip(snapshots,bounds[:-1])):
    x,seconds=create_features(cand,cutoff,events)
    features.append(x);costs.append(seconds)
    print(f'Snapshot {sid}: {len(x):,} pairs, {len(x.columns)} cols, {seconds:.2f}s')
all_data=pd.concat(features,ignore_index=True)
train=all_data.loc[all_data.snapshot.isin([0,1])].copy()
valid=all_data.loc[all_data.snapshot==2].copy()
test=all_data.loc[all_data.snapshot==3].copy()
print('Train / validation / test:',len(train),len(valid),len(test))
assert train.label.nunique()==valid.label.nunique()==test.label.nunique()==2


Snapshot 0: 46,272 pairs, 236 cols, 4.36s
Snapshot 1: 80,512 pairs, 236 cols, 5.79s
Snapshot 2: 83,936 pairs, 236 cols, 6.52s
Snapshot 3: 105,232 pairs, 236 cols, 7.97s
Train / validation / test: 126784 83936 105232


## Модели и поиск признаков на validation

Чтобы отделить полезность новых статистик от простого увеличения размерности, сравниваем: популярность, ручной baseline, расширенные ручные признаки и автоматические признаки. Ранжируем автоматически созданные признаки по importance модели, обученной **только на train**, а размер набора выбираем по validation. Тест до финального выбора не используется.

In [ ]:
from sklearn.metrics import roc_auc_score,log_loss

def ndcg_per_user(frame,pred,k=10):
    x=frame[['snapshot','user_id','label']].copy()
    x['score']=pred
    out=[]
    for (sid,uid),part in x.groupby(['snapshot','user_id'],sort=False):
        labels=part.sort_values('score',ascending=False).label.to_numpy()[:k]
        dcg=(labels/np.log2(np.arange(2,len(labels)+2))).sum()
        n=min(int(part.label.sum()),k)
        ideal=(1/np.log2(np.arange(2,n+2))).sum() if n else 0
        if ideal:out.append((sid,uid,dcg/ideal))
    return pd.DataFrame(out,columns=['snapshot','user_id','ndcg'])

def score(frame,pred):
    return {'ROC-AUC':roc_auc_score(frame.label,pred),
            'LogLoss':log_loss(frame.label,np.clip(pred,1e-7,1-1e-7)),
            'NDCG@10':ndcg_per_user(frame,pred).ndcg.mean()}

def fit(cols):
    m=CatBoostClassifier(iterations=200,depth=6,learning_rate=.07,l2_leaf_reg=5,
                         loss_function='Logloss',random_seed=SEED,verbose=False,
                         thread_count=2,allow_writing_files=False)
    t=time.monotonic()
    m.fit(train[cols],train.label,cat_features=[c for c in cols if train[c].dtype=='object'])
    return m,time.monotonic()-t

count_cols=[c for c in all_data if c.endswith(('_cnt','_nunique'))]
manual=['genre']+[c for c in count_cols if c.startswith('all_') or
                         (c.startswith(('w7_','w30_')) and c.endswith('_cnt'))]
manual=list(dict.fromkeys(manual))
all_features=['genre']+[c for c in all_data.columns if c not in
                          ['snapshot','user_id','item_id','label','genre']]
behavior_cols=[c for c in all_features if any(s in c for s in ['dur_','pct_','recency_days','_share'])]
manual_plus=list(dict.fromkeys(manual+[c for c in behavior_cols if c.startswith(('all_','w7_','w30_'))]))
assert len(behavior_cols)>0

rows=[];models={};sets={}
def evaluate(name,cols):
    m,seconds=fit(cols)
    v=score(valid,m.predict_proba(valid[cols])[:,1])
    rows.append({'model':name,'features':len(cols),'fit_s':seconds,**v})
    models[name]=m;sets[name]=cols
    print(name,len(cols),round(v['NDCG@10'],5),'fit_s',round(seconds,1))

evaluate('Popularity',['all_item_cnt'])
evaluate('Manual',manual)
evaluate('Manual_plus_behavior',manual_plus)
evaluate('Auto_all',all_features)
imp=pd.Series(models['Auto_all'].get_feature_importance(),index=all_features).drop('genre',errors='ignore').sort_values(ascending=False)
for k in [8,16,32,64,96]:
    evaluate(f'Auto_top{k}',['genre']+imp.head(k).index.tolist())
validation=pd.DataFrame(rows).sort_values('NDCG@10',ascending=False)
display(validation)
validation.to_csv(OUT/'validation_metrics.csv',index=False)
imp.to_csv(OUT/'feature_importance.csv',header=['importance'])
print('Генерация признаков, секунды:',costs)


Popularity 1 0.66945 fit_s 6.1
Manual 17 0.71249 fit_s 21.7
Manual_plus_behavior 92 0.71438 fit_s 33.9
Auto_all 232 0.71284 fit_s 54.3
Auto_top8 9 0.70547 fit_s 21.4
Auto_top16 17 0.70229 fit_s 23.7
Auto_top32 33 0.71482 fit_s 25.6
Auto_top64 65 0.71635 fit_s 29.7
Auto_top96 97 0.71365 fit_s 35.3


,model,features,fit_s,ROC-AUC,LogLoss,NDCG@10
7,Auto_top64,65,29.693509,0.877383,0.152766,0.716346
6,Auto_top32,33,25.605448,0.877287,0.153491,0.714819
2,Manual_plus_behavior,92,33.870616,0.876587,0.153218,0.714376
8,Auto_top96,97,35.324879,0.877093,0.153498,0.713655
3,Auto_all,232,54.338723,0.878562,0.153312,0.712840
1,Manual,17,21.657281,0.876158,0.153243,0.712488
4,Auto_top8,9,21.407919,0.866266,0.156659,0.705474
5,Auto_top16,17,23.688528,0.868797,0.156929,0.702292
0,Popularity,1,6.106071,0.837827,0.170592,0.669452


Генерация признаков, секунды: [4.364133224, 5.785353858999997, 6.521596330999955, 7.973158353000031]


## Итоговая оценка на test

Выбираем лучший **автоматический** набор только по validation; сравниваем с ручным baseline и вариантом с поведенческими признаками. Bootstrap ресэмплирует пользователей и оценивает неопределённость разницы средних NDCG@10.

In [ ]:
best_auto=validation.loc[validation.model.str.startswith('Auto_')].iloc[0]['model']
final_names=['Popularity','Manual','Manual_plus_behavior',best_auto]
final_rows=[];per_user={}
for name in dict.fromkeys(final_names):
    pred=models[name].predict_proba(test[sets[name]])[:,1]
    final_rows.append({'model':name,'features':len(sets[name]),**score(test,pred)})
    per_user[name]=ndcg_per_user(test,pred).set_index(['snapshot','user_id']).ndcg
final=pd.DataFrame(final_rows)
display(final)
final.to_csv(OUT/'test_metrics.csv',index=False)

# Парный bootstrap по пользователям
rng=np.random.default_rng(2026)
a=per_user[best_auto]; b=per_user['Manual']; common=a.index.intersection(b.index)
delta=(a.loc[common]-b.loc[common]).to_numpy()
boot=np.array([rng.choice(delta,size=len(delta),replace=True).mean() for _ in range(1500)])
ci=np.quantile(boot,[.025,.975])
print('Auto minus Manual:',round(delta.mean(),6),'95% CI:',np.round(ci,6),'n_users:',len(delta))
pd.DataFrame([{'best_auto':best_auto,'delta_ndcg':delta.mean(),
               'ci_low':ci[0],'ci_high':ci[1],'n_users':len(delta)}]).to_csv(OUT/'bootstrap.csv',index=False)


,model,features,ROC-AUC,LogLoss,NDCG@10
0,Popularity,1,0.865428,0.170222,0.664845
1,Manual,17,0.892302,0.149958,0.728725
2,Manual_plus_behavior,92,0.890495,0.151863,0.728283
3,Auto_top64,65,0.889302,0.151642,0.729137


Auto minus Manual: 0.000412 95% CI: [-0.003411  0.004148] n_users: 2740


## Диагностика и воспроизводимость

Проверяем важность семейств признаков и сохраняем параметры. Это описательный анализ обученной модели, а не новый цикл подбора по test.

In [ ]:
def family(c):
    if 'recency_days' in c:return 'recency'
    if '_share' in c:return 'share'
    if any(x in c for x in ['dur_','pct_']):return 'behavior'
    if c.endswith(('_cnt','_nunique')):return 'counts'
    return 'categorical'
family_table=(imp.rename('importance').reset_index().rename(columns={'index':'feature'})
              .assign(family=lambda d:d.feature.map(family))
              .groupby('family',as_index=False).agg(n_features=('feature','size'),total_importance=('importance','sum')))
display(family_table)
family_table.to_csv(OUT/'importance_by_family.csv',index=False)
config={'seed':SEED,'users':N_USERS,'negatives_per_positive':NEG_PER_POS,
        'windows':WINDOWS,'selected_auto':best_auto,'selected_features':sets[best_auto],
        'catboost_iterations':200,'catboost_depth':6,'catboost_learning_rate':.07,
        'feature_generation_seconds':costs,
        'limitations':['last_watch_dt is aggregated','sampled unexposed negatives','future-active user conditioning']}
(OUT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Артефакты:',*[p.name for p in OUT.iterdir()],sep='\n - ')


,family,n_features,total_importance
0,behavior,140,58.411266
1,counts,56,30.073981
2,recency,28,9.217887
3,share,7,2.296866


Артефакты:
 - config.json
 - validation_metrics.csv
 - importance_by_family.csv
 - feature_importance.csv
 - bootstrap.csv
 - test_metrics.csv


## Интерпретация результатов

Результаты следует интерпретировать с учётом доверительных интервалов, состава кандидатов и ограничений исходных данных.


## 8. Автоматический поиск семейств признаков (validation only)

В отличие от простого отбора top-K по важности, здесь алгоритм ищет **группы преобразований**. Начиная с ручного набора, он по очереди пробует добавить каждое семейство, выбирает максимальный прирост NDCG@10 на validation и останавливается, когда прирост меньше `MIN_GAIN`. Это *жадный* поиск, не ИИ-агент. Качество test при выборе семейств не используется. Число итераций и порог фиксируются заранее.

**Важно:** при поиске используется та же validation неоднократно, поэтому возможна подгонка к ней. Test оценивается только после завершения поиска.

In [ ]:
# Неизменяемые настройки отбора, объявленные до просмотра новых test-результатов
MIN_GAIN = 0.0005
MAX_FAMILIES = 5

def feature_family(col):
    if col == 'genre': return 'categorical'
    if '_share' in col: return 'relative_share'
    if '_recency_days' in col: return 'recency'
    if '_dur_' in col or '_pct_' in col: return 'behavior'
    if col.endswith(('_cnt','_nunique')): return 'counts'
    return 'other'

families = {fam: [c for c in all_features if feature_family(c)==fam and c not in manual]
            for fam in ['relative_share','recency','behavior','counts','other']}
families = {k:v for k,v in families.items() if v}
selected_cols = list(manual)
selected_families = []
current_score = float(validation.loc[validation.model=='Manual','NDCG@10'].iloc[0])
search_log = []
search_models = {}
for step in range(MAX_FAMILIES):
    options = []
    for fam, cols in families.items():
        if fam in selected_families: continue
        proposed = list(dict.fromkeys(selected_cols+cols))
        model, seconds = fit(proposed)
        val_ndcg = score(valid,model.predict_proba(valid[proposed])[:,1])['NDCG@10']
        options.append((val_ndcg,fam,proposed,model,seconds))
        search_log.append({'step':step+1,'family':fam,'n_features':len(proposed),
                           'validation_ndcg':val_ndcg,'fit_seconds':seconds,
                           'accepted':False})
    if not options: break
    winner = max(options,key=lambda t:t[0])
    gain = winner[0]-current_score
    if gain < MIN_GAIN:
        print(f'Stopping: best additional gain {gain:+.6f} < {MIN_GAIN}')
        break
    current_score, fam, selected_cols, selected_model, seconds = winner
    selected_families.append(fam)
    search_log[-len(options)+[x[1] for x in options].index(fam)]['accepted']=True
    print(f'Accepted {fam}, validation NDCG@10={current_score:.6f}, gain={gain:+.6f}')

search_table=pd.DataFrame(search_log)
display(search_table)
search_table.to_csv(OUT/'family_search_validation.csv',index=False)
print('Selected families:',selected_families,'features:',len(selected_cols))
# Сохраняем модель поиска, либо исходную Manual при отсутствии улучшений.
if selected_families:
    family_model=selected_model
else:
    family_model=models['Manual']
family_pred=family_model.predict_proba(test[selected_cols])[:,1]
family_test=score(test,family_pred)
print('FamilySearch TEST (one-time evaluation):',family_test)
pd.DataFrame([{'model':'FamilySearch','seed':SEED,'families':','.join(selected_families),
               'n_features':len(selected_cols),**family_test}]).to_csv(OUT/'family_search_test.csv',index=False)


Accepted counts, validation NDCG@10=0.714972, gain=+0.002484
Stopping: best additional gain +0.000326 < 0.0005


,step,family,n_features,validation_ndcg,fit_seconds,accepted
0,1,relative_share,24,0.712986,23.069426,False
1,1,recency,45,0.713241,25.585713,False
2,1,behavior,157,0.714044,43.893026,False
3,1,counts,57,0.714972,26.352495,True
4,2,relative_share,64,0.715298,27.376104,False
5,2,recency,85,0.713961,30.313379,False
6,2,behavior,197,0.715050,48.377556,False


Selected families: ['counts'] features: 57
FamilySearch TEST (one-time evaluation): {'ROC-AUC': np.float64(0.8910296999277657), 'LogLoss': 0.15053568226550643, 'NDCG@10': np.float64(0.726470102022217)}


## 9. Сложные отрицательные кандидаты (проверка устойчивости)

Создаём отдельный **test-only** набор кандидатов: те же положительные пары из последнего временного окна, но отрицательные фильмы выбираются с вероятностью, пропорциональной их исторической популярности. Популярность считается **строго до test cutoff**. Так сложнее опираться только на популярность. Модели не дообучаются и не перенастраиваются по этой проверке.

**Ограничение:** чтобы не присвоить отрицательный label фильму, который пользователь действительно посмотрел в целевом окне, генератор исключает положительные объекты этого окна. Это использование будущего **для построения offline evaluation**, но не для признаков или весов модели. Полученный набор кандидатов не является имитацией реального показа. Значения NDCG на random и hard наборах не сравниваются как единая метрика качества.

In [ ]:
def make_popularity_hard_candidates(history,future,snap_id,max_negs=NEG_PER_POS):
    rng=np.random.default_rng(SEED+9000+snap_id)
    popularity=history.groupby('item_id').size().sort_index()
    catalog=popularity.index.to_numpy()
    weights=np.power(popularity.to_numpy(dtype=float),0.75)
    known_users=set(history.user_id.unique())
    active=future[future.user_id.isin(known_users)]
    seen=history.groupby('user_id').item_id.agg(set).to_dict()
    available=set(catalog)
    positives=(active[active.item_id.isin(available)]
               .drop_duplicates(['user_id','item_id'])
               .sort_values(['user_id','ts']))
    rows=[]
    for uid,grp in positives.groupby('user_id',sort=False):
        pos_ids=grp.item_id.to_numpy()[:MAX_POS_PER_USER]
        blocked=seen[uid] | set(grp.item_id.tolist())
        rows.extend((snap_id,uid,int(item),1) for item in pos_ids)
        mask=np.array([item not in blocked for item in catalog])
        eligible=catalog[mask]
        nneg=min(len(eligible),max_negs*len(pos_ids))
        if nneg:
            prob=weights[mask]/weights[mask].sum()
            neg=rng.choice(eligible,size=nneg,replace=False,p=prob)
            rows.extend((snap_id,uid,int(item),0) for item in neg)
    c=pd.DataFrame(rows,columns=['snapshot','user_id','item_id','label'])
    c=c.merge(meta,on='item_id',how='left')
    c['genre']=c['genre'].fillna('unknown')
    return c

sid=3
cutoff,next_cutoff=bounds[sid:sid+2]
hist=events[events.ts<cutoff]
fut=events[(events.ts>=cutoff)&(events.ts<=next_cutoff)]
hard_candidates=make_popularity_hard_candidates(hist,fut,sid)
assert not hard_candidates.duplicated(['snapshot','user_id','item_id']).any()
# Положительные пары и пользователи совпадают, меняются только отрицательные кандидаты.
original_pos=set(zip(snapshots[sid].loc[snapshots[sid].label==1,'user_id'],
                     snapshots[sid].loc[snapshots[sid].label==1,'item_id']))
hard_pos=set(zip(hard_candidates.loc[hard_candidates.label==1,'user_id'],
                 hard_candidates.loc[hard_candidates.label==1,'item_id']))
assert original_pos==hard_pos
hard_test,hard_feature_seconds=create_features(hard_candidates,cutoff,events)
compare_models={'Popularity':(models['Popularity'],sets['Popularity']),
                'Manual':(models['Manual'],sets['Manual']),
                'Manual_plus_behavior':(models['Manual_plus_behavior'],sets['Manual_plus_behavior']),
                'FamilySearch':(family_model,selected_cols)}
hard_rows=[]
for name,(model,cols) in compare_models.items():
    pred=model.predict_proba(hard_test[cols])[:,1]
    hard_rows.append({'model':name,'candidate_protocol':'popularity_weighted_test',
                      'seed':SEED,'features':len(cols),**score(hard_test,pred)})
hard_metrics=pd.DataFrame(hard_rows)
display(hard_metrics)
hard_metrics.to_csv(OUT/'hard_candidates_test_metrics.csv',index=False)
print('Hard candidate feature generation seconds:',round(hard_feature_seconds,2))


,model,candidate_protocol,seed,features,ROC-AUC,LogLoss,NDCG@10
0,Popularity,popularity_weighted_test,4049,1,0.629242,0.402773,0.340534
1,Manual,popularity_weighted_test,4049,17,0.705094,0.347039,0.407430
2,Manual_plus_behavior,popularity_weighted_test,4049,92,0.703646,0.352883,0.408162
3,FamilySearch,popularity_weighted_test,4049,57,0.704135,0.344025,0.405954


Hard candidate feature generation seconds: 8.62


## 10. Аудит и экспорт

Сохраняем параметры, количество кандидатов и сравнительные результаты. : если гипотеза возникла из результатов test, проверяйте её на новом независимом эксперименте. Измерение масштабируемости на 5k/15k/30k пользователей не входит в этот ноутбук; оно требует отдельного запуска с разными `N_USERS`. Не заявляйте, что такой эксперимент уже проведён.

In [ ]:
audit={
 'seed':SEED,'n_users_requested':N_USERS,'n_events_sampled':len(events),
 'n_auto_features':len(all_features),'manual_features':len(manual),
 'selected_families':selected_families,'selected_feature_count':len(selected_cols),
 'validation_min_gain':MIN_GAIN,'max_families':MAX_FAMILIES,
 'train_candidates':len(train),'validation_candidates':len(valid),'test_candidates':len(test),
 'hard_test_candidates':len(hard_test),'feature_generation_seconds':costs,
 'hard_test_feature_generation_seconds':hard_feature_seconds,
 'limitations':['KION last_watch_dt is not full watch event history',
                'future positives used only for offline candidate labeling/exclusion',
                'test user cohort selected by activity in target interval',
                'random vs popularity-weighted candidates define different tasks']}
with open(OUT/'audit_and_config.json','w',encoding='utf-8') as f:
    json.dump(audit,f,ensure_ascii=False,indent=2,default=str)
print('Output directory:',OUT)
for file in sorted(OUT.glob('*')):print(file.name)
# Скачать все результаты одним ZIP (не требует доступа к Google Drive)
import shutil
archive_path=shutil.make_archive('/content/AutoFeatureRec_extended_results','zip',root_dir=OUT)
print('Results ZIP:',archive_path)
# В Colab при необходимости: from google.colab import files; files.download(archive_path)


Output directory: /content/autofeaturerec_final_2026
audit_and_config.json
bootstrap.csv
config.json
family_search_test.csv
family_search_validation.csv
feature_importance.csv
hard_candidates_test_metrics.csv
importance_by_family.csv
test_metrics.csv
validation_metrics.csv
Results ZIP: /content/AutoFeatureRec_extended_results.zip
